## Import Libraries

### Load Datasets and Display Summary Info


In [110]:
import pandas as pd 
import numpy as np 
df_billing=pd.read_csv("Billing_Data.csv")
df_patient=pd.read_csv("Patient_Data.csv")

print("--- Patient Dataset Info ---")
df_patient.info()

print("\n--- Billing Dataset Info ---")
df_billing.info()

--- Patient Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 464.0+ bytes

--- Billing Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   PatientID         5 non-null      int64
 1   InsuranceCovered  5 non-null      int64
 2   FinalAmount       5 non-null      int64
dtypes: int64(3)
memory usage: 248.0 bytes


In [111]:
df_billing

,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


In [112]:
df_patient

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45
5,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00


### Task:2 2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [113]:
df_patient[['PatientID', 'Department', 'Doctor', 'BillAmount']]

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


In [114]:
df_patient

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45
5,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00


### 3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

In [115]:
# Drop administrative columns from the original patient dataframe
admin_cols = ["ReceptionistID", "CheckInTime"]
df_patient = df_patient.drop(columns=admin_cols, errors="ignore")

print("Remaining columns in df_patient:", df_patient.columns.tolist())

Remaining columns in df_patient: ['PatientID', 'Name', 'Department', 'Doctor', 'BillAmount']


In [116]:
df_patient=df_patient[['PatientID','Department','Doctor','BillAmount']]

### 4.	Use groupby to find total bill amount per department.

In [117]:
# To group by department and find BillAmount, we temporarily link them for this task
department_total_bill = (
    df_patient.groupby("Department")["BillAmount"].sum().reset_index()
)

print("--- Total Bill Amount Per Department ---")
print(department_total_bill)

--- Total Bill Amount Per Department ---
    Department  BillAmount
0   Cardiology     16200.0
1  Dermatology         0.0
2    Neurology         0.0
3  Orthopedics      7500.0


In [118]:
department_revenue = (
    df_patient.groupby("Department")["BillAmount"].sum().reset_index()
)

print("Total Bill Amount (Revenue) Per Department:")
print(department_revenue)

Total Bill Amount (Revenue) Per Department:
    Department  BillAmount
0   Cardiology     16200.0
1  Dermatology         0.0
2    Neurology         0.0
3  Orthopedics      7500.0


### 5.	Remove duplicate patient records based on PatientID.


In [119]:
# Remove duplicate records from the patient dataset
df_patient = df_patient.drop_duplicates(subset=["PatientID"])

print(f"Duplicates removed. Unique patients count: {df_patient.shape[0]}")

Duplicates removed. Unique patients count: 5


### 6.	Fill missing BillAmount values with the mean bill amount.

In [120]:
# Calculate the mean from the billing dataset
mean_bill = df_patient["BillAmount"].mean()
print(f"Calculated Mean Bill Amount: ₹{mean_bill:.2f}")

# Fill missing values in the billing dataset
df_patient["BillAmount"] = df_patient["BillAmount"].fillna(mean_bill)

print(
    "Missing values remaining in df_billing:",
    df_patient["BillAmount"].isna().sum(),
)

Calculated Mean Bill Amount: ₹6233.33
Missing values remaining in df_billing: 0


In [121]:
df_patient

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.000000
1,102,Neurology,Dr. John,6233.333333
2,103,Orthopedics,Dr. Lee,7500.000000
3,104,Cardiology,Dr. Smith,6200.000000
4,105,Dermatology,Dr. Rose,6233.333333


In [122]:
df_billing

,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


### 7.	Merge the billing dataset with patient dataset on PatientID.


In [123]:
# Final structural merge of the cleaned datasets
df_merged = pd.merge(df_patient, df_billing, on="PatientID", how="left")

print("--- Merged Dataset Preview ---")
print(df_merged.head())

--- Merged Dataset Preview ---
   PatientID   Department     Doctor   BillAmount  InsuranceCovered  \
0        101   Cardiology  Dr. Smith  5000.000000              2000   
1        102    Neurology   Dr. John  6233.333333              1500   
2        103  Orthopedics    Dr. Lee  7500.000000              2500   
3        104   Cardiology  Dr. Smith  6200.000000              3000   
4        105  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  


In [126]:
df_merged

,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Neurology,Dr. John,6233.333333,1500,3500
2,103,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Dermatology,Dr. Rose,6233.333333,1000,4000


### 8.	Concatenate an additional DataFrame that contains new patients for the current week (row-wise).


In [135]:
# New patient records data for the current week
new_patients_data = {
    "PatientID": [301, 302],
    "Department": ["Cardiology", "Neurology"],
    "Doctor": ["Dr. Smith", "Dr. Adams"],
    "BillAmount": [5500.0, 7200.0],
    "InsuranceCovered":[2000,1500],
    'FinalAmount':[3500,5700]
  
}
df_new_patients = pd.DataFrame(new_patients_data)

# Perform row-wise concatenation
df_final_rows = pd.concat([df_merged, df_new_patients], ignore_index=True)

print(f"Total rows after adding new weekly patients: {df_final_rows.shape[0]}")

Total rows after adding new weekly patients: 7


In [136]:
df_final_rows

,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Neurology,Dr. John,6233.333333,1500,3500
2,103,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Dermatology,Dr. Rose,6233.333333,1000,4000
5,301,Cardiology,Dr. Smith,5500.000000,2000,3500
6,302,Neurology,Dr. Adams,7200.000000,1500,5700


### 9.	Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).


In [137]:
insurance_covered = df_final_rows["BillAmount"] * 0.50
final_amount = df_final_rows["BillAmount"] - insurance_covered

df_new_categories = pd.DataFrame(
    {"InsuranceCovered": insurance_covered, "FinalAmount": final_amount}
)

df_final_cleaned_dataset = pd.concat([df_final_rows, df_new_categories], axis=1)

print("--- Final Cleaned Dataset (Task 9 Complete) ---")

--- Final Cleaned Dataset (Task 9 Complete) ---


In [138]:
df_final_cleaned_dataset

,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.000000,2000,3000,2500.000000,2500.000000
1,102,Neurology,Dr. John,6233.333333,1500,3500,3116.666667,3116.666667
2,103,Orthopedics,Dr. Lee,7500.000000,2500,5000,3750.000000,3750.000000
3,104,Cardiology,Dr. Smith,6200.000000,3000,3200,3100.000000,3100.000000
4,105,Dermatology,Dr. Rose,6233.333333,1000,4000,3116.666667,3116.666667
5,301,Cardiology,Dr. Smith,5500.000000,2000,3500,2750.000000,2750.000000
6,302,Neurology,Dr. Adams,7200.000000,1500,5700,3600.000000,3600.000000
